# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
#general_kb = build_general_kb(n, box_h, box_w, givens)
#r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
#query = atom('Is', r, c, v)

#print(pl_resolution(general_kb, query))    
#print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


**OBSERVATION**
Two separate tests were run on a single cell/value pair from the general KB. First, pl_resolution and tt_entails were run together (both uncommented at once); neither produced a result after more than one minute. To rule out any interference between the two calls, each algorithm was then tested in isolation — with only one of the two lines uncommented at a time — and both pl_resolution and tt_entails independently continued running past 30 seconds without returning a result, requiring manual interruption (Kernel > Interrupt) in every case. This confirms that neither algorithm is impractically slow merely because of running alongside the other — each one alone is already unable to complete even this single, minimal query within a reasonable timeframe.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.

### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

 # Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

 # Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")

solve_full_grid_fc: 2.74s
solve_full_grid_bc: 8.45s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**
a) In the general knowledge base, each of Sudoku's rules was translated directly into one or more CNF clauses, since PropKB places no restriction on how many positive or negative literals a clause may contain.

The at-least-one-value constraint was encoded, for every cell (r, c), as a single disjunctive clause spanning all n possible values: (Is_r_c_1 | Is_r_c_2 | ... | Is_r_c_n). This clause is generated once per cell across the entire grid, guaranteeing that every cell is assigned at least one value.

The at-most-one-value constraint was encoded using the negation of two candidate values within the same cell: ~Is_r_c_v1 | ~Is_r_c_v2. This clause states that a cell cannot simultaneously hold two different values, v1 and v2. Since this restriction must hold for every possible pair of values, one such clause was generated for every pair (v1, v2) within a cell, and this process was repeated for every cell in the grid.

Row, column, and box uniqueness were encoded using the same negation logic, but applied between two different cells rather than two different values. For every cell (r, c) and each of its peers (pr, pc) — any cell sharing the same row, column, or box — a clause of the form ~Is_r_c_v | ~Is_pr_pc_v was generated for every value v from 1 to n. This ensures that no two cells sharing a row, column, or box can hold the same value, regardless of which value is being checked.

Finally, the given cells were encoded directly as unit clauses, Is_r_c_v, asserting that each given cell's stated value is simply true, with no disjunction or negation required.

Taken together, these four clause types form a complete and direct translation of Sudoku's constraints into propositional logic. This was possible without any restructuring because general clauses impose no restriction on how many positive or negative literals a clause may contain, allowing each rule to be expressed in whatever clause form most naturally matched its own logical structure.

b) Unlike a general clause, a definite (Horn) clause permits at most one positive literal. This restriction rules out the direct at-least-one-value clause used in the general KB, since it contains n positive Is literals in a single disjunction. To work around this, a second family of atoms, Not_r_c_v, was introduced to represent negative information — the claim that a cell does not hold a particular value. This makes it possible to accumulate what a cell cannot be through valid Horn clauses, and to conclude its actual value only once every other candidate has been eliminated.

The elimination rule captures this negative information directly: atom('Is', pr, pc, v) |'==>'| atom('Not', r, c, v). This states that if any peer of a cell — a cell sharing the same row, column, or box — holds value v, then the cell itself cannot hold v. This rule is generated for every cell, for every peer of that cell, and for every value v, ensuring that every relevant elimination is captured across the grid.

The last-candidate rule then uses this accumulated negative information to derive a cell's actual value. For a given cell and a candidate value v_target, the Not_r_c_v facts for every other value (v ≠ v_target) are gathered and combined into a single conjunction. This conjunction implies Is_r_c_v_target: once every value other than v_target has been ruled out, v_target must be the cell's value. This rule is generated for every possible v_target, for every cell, so that the correct value is derivable regardless of which one it turns out to be.

Finally, the given cells are encoded directly as unit clauses, Is_r_c_v, asserting that each given cell's stated value is simply true.

Together, the elimination and last-candidate rules recreate the effect of the general KB's at-least-one and uniqueness constraints indirectly, using only clauses with a single positive literal — satisfying the structural restriction of a definite clause while preserving the same underlying Sudoku logic.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**
No, I would not use either pl_resolution or tt_entails as the default algorithm for solving Sudoku puzzles. Both methods are sound and complete, but their cost grows out of control well before they can be applied to a puzzle of this size.

tt_entails works by checking every possible True/False assignment to every propositional symbol involved. For a 9×9 Sudoku puzzle, there are 9 rows × 9 columns × 9 values = 729 possible Is_r_c_v symbols, meaning tt_entails would need to check up to 2^729 combinations to determine entailment — a number so large it is computationally infeasible, even for a single query.

pl_resolution considers every pair of clauses in the current clause set to look for resolvents, which is already an O(n²) operation for a single round. Since new resolvent clauses are added back into the set, each subsequent round works over an even larger set of clauses, compounding the cost round over round as the algorithm searches for a contradiction.

This matches what was observed directly: when tested on a single cell/value pair, both pl_resolution and tt_entails ran for more than 30 seconds without returning a result, whether run together or in isolation. Since solving the full puzzle would require this exact reasoning repeated across every cell and every candidate value, neither algorithm is a practical choice for actually solving Sudoku, despite both being theoretically guaranteed to produce a correct answer.

Forward and backward chaining are a better fit for a problem of this size, since they only ever pursue rules whose premises are already established, rather than exhaustively enumerating truth assignments or resolving every possible pair of clauses. This targeted approach avoids the combinatorial blow-up seen above, allowing the full puzzle to be solved in a practical amount of time.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**
Function of pl_bc_entails(kb, query)

    // Permanent memory -- only ever holds proven-True facts, and is kept across every retry attempt below
    permanent_cache = {}

    // One single search attempt for question 'q'. It shares 'permanent_cache' with every other attempt, but 'round_memo' is thrown away and rebuilt fresh at the         // start of each new attempt.
    Function bc_once(q, visiting, round_memo)

        If q is already in permanent_cache
            Return True    // proven true in an earlier attempt

        If q is already in round_memo
            Return round_memo[q]   // already worked out earlier in THIS attempt

        If q is already being worked on (in visiting)
            Return False    // loop detected; may still succeed on a later retry

        Mark q as being worked on
        result = False

        // Base case: is this simply a given fact already in the KB?
        Check if q is a known fact in the kb:
            If it is: result = True

        // Otherwise, look for a rule that could prove this query
        If result is not yet True:
            For each rule in the kb, check (premises => conclusion),
            
                // OR across rules -- if this rule fails, try the next
                If conclusion == q:
                
                    // Recursion -- each premise is a new, smaller question, asked via bc_once again AND across premises -- every one must succeed
                    If bc_once(premise, visiting, round_memo) is true
                       for every premise in premises:
                        result = True
                        break

        Un-mark q as being worked on
        round_memo[q] = result             // remember for the REST of this attempt
        If result is True:
            permanent_cache[q] = True      // remember PERMANENTLY, across all attempts
        Return result

    // Retry the whole search from scratch (fresh visiting, fresh round_memo) as many times as needed, but always keeping every permanently-proven fact so far. Stop      // once the query succeeds, or once a full retry adds nothing new (a fixed point).
    Loop forever:
        count_before = size of permanent_cache
        found = bc_once(query, empty set, empty dict)
        If found or size of permanent_cache == count_before:
            Return found

Design Challenges 
1) Termination: Because Sudoku's peer relationships are circular (a cell's value can depend on a peer's value, which can depend back on the original cell), a query could recurse into itself forever. The fix was a "currently being worked on" check: if a query is asked again while it's still being resolved further up the same chain, the algorithm gives up on that one path and returns False, without exploring it endlessly.

2) Correctness: An early version returned False on a repeated query and permanently cached it, which caused an AssertionError — some facts were actually True through a different, non-circular path, but the premature False had already been locked in. The fix was to separate "currently being worked on" (temporary, used only to break loops) from the cache (permanent, written only once the function has fully finished checking every fact and rule for that query) — so a loop only blocks one dead-end path, without deciding the final answer too early.

3) Completeness across the whole search: A subtler issue emerged from order-dependence: within one search attempt, a fact needed early in the search can depend on evidence that is only discovered later in that same attempt, since Sudoku cells share many overlapping peers. By the time that evidence exists, the earlier check has already run and failed. The fix was to retry the whole search repeatedly, permanently keeping every fact already proven True so later attempts can build on it, and stopping once a retry proves nothing new

4) Efficiency: Retrying introduced a new problem: only True answers can be trusted permanently across attempts (challenge 3 relies on this), but without remembering anything within a single attempt, the same sub-question would get re-derived from scratch every time it came up — and Sudoku's overlapping peers mean it comes up often, reintroducing the original slowdown. The fix was a second, temporary memory that remembers both True and False answers, but only for the duration of one attempt, discarded and rebuilt fresh at the start of the next. Separately, before even asking pl_bc_entails about a cell's candidate values, the solver also excludes any value already ruled out by a peer's given value — since basic Sudoku uniqueness guarantees the true answer can never match one, avoiding an expensive search to confirm something the puzzle's structure already tells us for free.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**
# Suppose cells A = (rA, cA) and B = (rB, cB) are a "naked pair" for
# values v1 and v2, both sitting in the same unit (row, column, or box).

# "Cell A can only be v1 or v2" = every OTHER value has been eliminated for A
A_premises = [atom('Not', rA, cA, v) for v in range(1, n + 1) if v not in (v1, v2)]

# Same idea for cell B
B_premises = [atom('Not', rB, cB, v) for v in range(1, n + 1) if v not in (v1, v2)]

# Both sets of premises are needed together
all_premises = A_premises + B_premises

# For every OTHER cell (r, c) sharing that same unit with A and B...
for (r, c) in unit_cells:
    if (r, c) != (rA, cA) and (r, c) != (rB, cB):
        # ...it can't be v1...
        kb.tell(associate('&', all_premises) |'==>'| atom('Not', r, c, v1))
        # ...and, as a SEPARATE clause, it can't be v2 either
        kb.tell(associate('&', all_premises) |'==>'| atom('Not', r, c, v2))

Consequences 
Naked Pairs fits Horn logic in principle: "cell A can only be one of two values" is just a conjunction of Not facts — every other value already eliminated — and a conjunction of positive premises implying one positive conclusion is exactly the form PropDefiniteKB.tell() requires.

The cost shows up once you try to actually build it. Propositional logic has no way to write a single rule that applies to "any pair of cells" the way a variable would in first-order logic — every specific unit, every specific pair of cells within it, and every specific pair of values needs its own separately written-out clause. So instead of adding one rule, this technique turns into a large family of near-identical rules repeated across every row, column, and box, substantially growing the size of the KB beyond the base Sudoku constraints.

The real limit this exposes: how much an encoding grows depends on how many cells a rule needs to look at jointly. A single-cell rule needs one clause per cell/value combination, but a two-cell technique like Naked Pairs needs one clause per pair of cells and pair of values, which grows much faster. Horn logic can state the rule correctly — the limit is whether a purely propositional encoding stays practical as the pattern being captured grows more complex.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**
(a) When backward chaining is significantly faster
Backward chaining wins when a large KB needs to answer a query that only depends on a small, targeted subset of it — it explores just what's relevant to that goal, while forward chaining derives the entire KB's consequences regardless. A single targeted query, like checking one specific cell/value, is the clearest example.

(b) When it offers no advantage, or performs worse
That advantage disappears when most of the KB is relevant to the query, or when the query isn't derivable — proving a negative means exhausting every proof path, which can cost more than forward chaining deriving everything once and checking membership. Solving the entire grid is this case: since every cell needs a value, backward chaining ends up touching nearly every fact anyway, same as forward chaining.

Time Taken for Sudoku Solver.
Forward Chaining - 2.74
Backward Chaining - 8.45 

Backward chaining took roughly three times longer than forward chaining here, contrary to the usual expectation that goal-directed search is faster. This is because Sudoku's constraints are densely interconnected, so proving one cell's value typically requires chaining through many other cells rather than a small relevant subset — and the retry mechanism needed for completeness adds further overhead whenever a guessed value turns out to be False. Backward chaining therefore loses its usual efficiency advantage in this densely connected knowledge base, even after indexing removed the linear clause-scan overhead from both algorithms."

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
